## CELL 0: GLOBAL SEED FIX

In [ ]:
# ==============================================================================
# CELL 0: GLOBAL SEED FIX (REPRODUCIBILITY)
# ==============================================================================

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    print(f"[*] Locking random seeds to {seed}...")
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("✅ Seeds locked. Pipeline is reproducible.")

seed_everything(seed=42)

## CELL 1: BINARY CONVERTER + DATASET PIPELINE

In [ ]:
# ==============================================================================
# CELL 1: BINARY CONVERTER + DATASET PIPELINE
#
# BinaryConverter  : ONE-TIME CSV → float32 .npy conversion.
# PhysiologicalDataset : Reads pre-converted binaries (zero CSV overhead).
# build_cv_pools   : Pools train+val for StratifiedKFold CV.
#
# Ablation-aware: dataset accepts `channels` list so the loader can
# return only the requested modalities.
# ==============================================================================

import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from scipy.signal import find_peaks, peak_widths
import warnings
warnings.filterwarnings('ignore')


# ─────────────────────────────────────────────────────────────────────────────
# BinaryConverter — run ONCE to pre-process CSVs into .npy files
# ─────────────────────────────────────────────────────────────────────────────
class BinaryConverter:
    """
    Converts the CSV-based dataset into memory-efficient float32 .npy binaries.

    Output layout:
        <binary_root>/<split>/abp/<window>/<stem>.npy       ← ABP waveform (3750,)
        <binary_root>/<split>/ppg/<window>/<stem>.npy       ← PPG waveform (3750,)
        <binary_root>/<split>/ecg/<window>/<stem>.npy       ← ECG waveform (3750,)
        <binary_root>/<split>/meta/<window>/<stem>_meta.npz ← label + scalars (4,)

    Scalars stored (all divided by 100 for stability):
        [MAP, PP, PulseWidth, HRV]

    Usage:
        conv = BinaryConverter(data_dir='...', binary_root='...')
        conv.convert_all(horizons=[3,5,7,10,15], splits=['train','val','test'])
    """

    TARGET_LEN    = 3750
    FS            = 125
    MIN_PEAK_DIST = int(125 * 0.4)

    def __init__(self, data_dir: str, binary_root: str):
        self.data_dir    = data_dir
        self.binary_root = binary_root

    # ── Internal signal helpers ───────────────────────────────────────────────
    @staticmethod
    def _safe_extract_and_pad(df, target_len=3750) -> np.ndarray:
        val = df.iloc[0, 6:].values.astype(np.float32)
        if np.isnan(val).any():
            fill = np.nanmean(val)
            fill = float(fill) if not np.isnan(fill) else 0.0
            val  = np.nan_to_num(val, nan=fill)
        if len(val) >= target_len:
            return val[:target_len]
        return np.pad(val, (0, target_len - len(val)), mode='constant')

    @classmethod
    def _extract_hrv(cls, ecg_signal: np.ndarray) -> float:
        peaks, _ = find_peaks(ecg_signal, distance=cls.MIN_PEAK_DIST, prominence=0.5)
        if len(peaks) < 2:
            return 0.0
        rr = np.diff(peaks) / cls.FS * 1000
        return float(np.std(rr))

    @classmethod
    def _extract_pulse_width(cls, ppg_signal: np.ndarray) -> float:
        peaks, _ = find_peaks(ppg_signal, distance=cls.MIN_PEAK_DIST)
        if len(peaks) < 1:
            return 0.0
        widths, _, _, _ = peak_widths(ppg_signal, peaks, rel_height=0.5)
        return float(np.mean(widths)) / cls.FS * 1000

    @classmethod
    def _compute_scalars(cls, abp_raw, ppg_raw, ecg_raw) -> np.ndarray:
        map_val = float(np.mean(abp_raw))
        pp_val  = float(np.max(abp_raw) - np.min(abp_raw))
        if np.isnan(map_val) or np.isinf(map_val): map_val = 65.0
        if np.isnan(pp_val)  or np.isinf(pp_val):  pp_val  = 40.0
        pw_val  = cls._extract_pulse_width(ppg_raw)
        hrv_val = cls._extract_hrv(ecg_raw)
        return np.array([map_val / 100.0, pp_val / 100.0,
                         pw_val / 100.0,  hrv_val / 100.0], dtype=np.float32)

    # ── Per-horizon conversion ────────────────────────────────────────────────
    def _convert_one_horizon(self, split: str, horizon: int):
        window  = f'offset_{horizon}m'
        abp_dir = os.path.join(self.data_dir, split, 'abp', window)
        ppg_dir = os.path.join(self.data_dir, split, 'ppg', window)
        ecg_dir = os.path.join(self.data_dir, split, 'ecg', window)

        out_abp  = os.path.join(self.binary_root, split, 'abp',  window)
        out_ppg  = os.path.join(self.binary_root, split, 'ppg',  window)
        out_ecg  = os.path.join(self.binary_root, split, 'ecg',  window)
        out_meta = os.path.join(self.binary_root, split, 'meta', window)
        for d in [out_abp, out_ppg, out_ecg, out_meta]:
            os.makedirs(d, exist_ok=True)

        abp_files = sorted([f for f in os.listdir(abp_dir) if f.endswith('_wave.csv')])
        print(f"  [{split.upper()} | {window}] Converting {len(abp_files)} cases...")

        for fname in abp_files:
            stem  = fname.replace('_wave.csv', '')
            pname = fname.replace('abp', 'ppg')
            ename = fname.replace('abp', 'ecg')

            df_abp = pd.read_csv(os.path.join(abp_dir, fname))
            df_ppg = pd.read_csv(os.path.join(ppg_dir, pname))
            df_ecg = pd.read_csv(os.path.join(ecg_dir, ename))

            abp_raw = self._safe_extract_and_pad(df_abp)
            ppg_raw = self._safe_extract_and_pad(df_ppg)
            ecg_raw = self._safe_extract_and_pad(df_ecg)

            np.save(os.path.join(out_abp, f'{stem}.npy'), abp_raw)
            np.save(os.path.join(out_ppg, f'{stem}.npy'), ppg_raw)
            np.save(os.path.join(out_ecg, f'{stem}.npy'), ecg_raw)

            label   = int(df_abp.iloc[0]['label_binary'])
            scalars = self._compute_scalars(abp_raw, ppg_raw, ecg_raw)

            np.savez(
                os.path.join(out_meta, f'{stem}_meta.npz'),
                label=np.array([label], dtype=np.int8),
                scalars=scalars,
            )

    def convert_all(self, horizons: list, splits: list):
        for split in splits:
            for h in horizons:
                self._convert_one_horizon(split, h)
        print("\n[✓] Binary conversion complete.")


# ─────────────────────────────────────────────────────────────────────────────
# PhysiologicalDataset — binary-backed, ablation-aware
# ─────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ['ppg', 'abp', 'ecg']   # canonical order used in model input

class PhysiologicalDataset(Dataset):
    """
    Reads from pre-converted float32 .npy files.

    Args:
        binary_root : Root produced by BinaryConverter.convert_all().
        split       : 'train' | 'val' | 'test'
        horizon     : Prediction horizon in minutes.
        channels    : List of channel names to return, e.g. ['ppg','abp'].
                      Must be a subset of ['ppg','abp','ecg'].
        augment     : Whether to apply on-the-fly augmentation.
    """

    def __init__(self, binary_root: str, split='train', horizon=5,
                 channels=None, augment=False):
        super().__init__()
        self.split    = split
        self.horizon  = horizon
        self.augment  = augment
        self.channels = channels if channels is not None else ['ppg', 'abp', 'ecg']
        self.window   = f'offset_{horizon}m'

        # Validate channel names
        for ch in self.channels:
            assert ch in CHANNEL_NAMES, f"Unknown channel: {ch}"

        self.dirs = {
            'abp':  os.path.join(binary_root, split, 'abp',  self.window),
            'ppg':  os.path.join(binary_root, split, 'ppg',  self.window),
            'ecg':  os.path.join(binary_root, split, 'ecg',  self.window),
            'meta': os.path.join(binary_root, split, 'meta', self.window),
        }

        # Build stem list from abp dir (always present as anchor)
        anchor_dir = self.dirs['abp']
        self.stems = sorted([f.replace('.npy', '')
                             for f in os.listdir(anchor_dir)
                             if f.endswith('.npy')])

        self.labels = []
        for stem in self.stems:
            meta = np.load(os.path.join(self.dirs['meta'], f'{stem}_meta.npz'))
            self.labels.append(int(meta['label'][0]))

        pos_rate = sum(self.labels) / max(len(self.labels), 1)
        ch_str   = '+'.join(self.channels)
        print(f"  [{split.upper()} | horizon={horizon}m | ch={ch_str}] "
              f"{len(self.stems)} cases | pos_rate={pos_rate:.2f}")

    def __len__(self):
        return len(self.stems)

    @staticmethod
    def _zscore(x: np.ndarray) -> np.ndarray:
        return (x - x.mean()) / (x.std() + 1e-8)

    def _augment_signals(self, signals: dict) -> dict:
        """signals: dict of channel_name -> np.ndarray(3750,)"""
        abp = signals.get('abp')
        ppg = signals.get('ppg')
        ecg = signals.get('ecg')

        # Respiratory-like sine artefact on PPG/ECG
        if ppg is not None and ecg is not None and np.random.rand() < 0.5:
            t    = np.linspace(0, 3750 / 125, 3750)
            freq = np.random.uniform(0.1, 0.4)
            amp  = np.random.uniform(0.02, 0.08)
            wave = amp * np.sin(2 * np.pi * freq * t)
            ppg += wave
            ecg += wave

        # Amplitude scaling per channel
        for key in signals:
            if np.random.rand() < 0.5:
                signals[key] = signals[key] * np.random.uniform(0.90, 1.10)

        # Temporal shift (same shift for all channels)
        if np.random.rand() < 0.4:
            shift = np.random.randint(-25, 25)
            for key in signals:
                signals[key] = np.roll(signals[key], shift)

        # Random channel mask-out
        if np.random.rand() < 0.3:
            cut   = np.random.randint(60, 200)
            start = np.random.randint(0, 3750 - cut)
            ch    = np.random.choice(list(signals.keys()))
            signals[ch][start:start + cut] = 0.0

        if abp is not None: signals['abp'] = abp
        if ppg is not None: signals['ppg'] = ppg
        if ecg is not None: signals['ecg'] = ecg
        return signals

    def __getitem__(self, idx):
        stem = self.stems[idx]

        # Load all required waveforms
        raw = {}
        for ch in self.channels:
            raw[ch] = np.load(os.path.join(self.dirs[ch], f'{stem}.npy'))

        meta    = np.load(os.path.join(self.dirs['meta'], f'{stem}_meta.npz'))
        label   = int(meta['label'][0])
        scalars = meta['scalars'].astype(np.float32)   # (4,) [MAP, PP, PW, HRV]

        # Z-score normalise per channel
        norm = {ch: self._zscore(raw[ch]) for ch in self.channels}

        if self.augment:
            norm = self._augment_signals(norm)

        # Stack channels in canonical order: ppg, abp, ecg (as present)
        channel_tensors = []
        for ch in CHANNEL_NAMES:
            if ch in norm:
                t = torch.from_numpy(norm[ch].astype(np.float32)).unsqueeze(0)  # (1, 3750)
                channel_tensors.append(t)

        waveform_t = torch.cat(channel_tensors, dim=0)   # (C, 3750)  C = len(channels)
        scalars_t  = torch.from_numpy(scalars)            # (4,)
        label_t    = torch.tensor(label, dtype=torch.float32)

        return (waveform_t, scalars_t), label_t


# ─────────────────────────────────────────────────────────────────────────────
# build_cv_pools
# ─────────────────────────────────────────────────────────────────────────────
def build_cv_pools(binary_root: str, horizon: int, channels: list,
                   n_folds: int = 5):
    """Build pooled train+val datasets and fold indices."""
    kw = dict(binary_root=binary_root, horizon=horizon, channels=channels)

    train_aug   = PhysiologicalDataset(**kw, split='train', augment=True)
    val_aug     = PhysiologicalDataset(**kw, split='val',   augment=True)
    train_noaug = PhysiologicalDataset(**kw, split='train', augment=False)
    val_noaug   = PhysiologicalDataset(**kw, split='val',   augment=False)

    pooled_aug   = ConcatDataset([train_aug,   val_aug])
    pooled_noaug = ConcatDataset([train_noaug, val_noaug])
    pooled_labels = train_aug.labels + val_aug.labels

    test_ds = PhysiologicalDataset(**kw, split='test', augment=False)

    skf   = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    folds = list(skf.split(np.zeros(len(pooled_labels)), pooled_labels))

    print(f"  Pooled CV: {len(pooled_labels)} cases | Test: {len(test_ds)} cases")
    return pooled_aug, pooled_noaug, pooled_labels, folds, test_ds


print("Cell 1 ready.")

## CELL 2: PatchTST MODEL — SIGMOID CLASSIFIER (NO MAP HEAD)

In [ ]:
# ==============================================================================
# CELL 2: PatchTST MODEL — SIGMOID BINARY CLASSIFIER
#
# Changes vs. previous version:
#   • MAP regression head removed entirely.
#   • Cosine-similarity classifier replaced with a linear → Sigmoid head.
#   • n_channels is now dynamic (1, 2, or 3) to support channel ablation.
#   • scalar_dim kept as 4 (MAP, PP, PW, HRV) from BinaryConverter.
# ==============================================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────────────────────
# Positional Encoding
# ─────────────────────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):          # x: (B, L, D)
        return self.dropout(x + self.pe[:, :x.size(1)])


# ─────────────────────────────────────────────────────────────────────────────
# Channel-Independent PatchTST Encoder
# ─────────────────────────────────────────────────────────────────────────────
class PatchTSTEncoder(nn.Module):
    """
    Channel-independent PatchTST backbone.

    Each channel's time-series is split into overlapping patches,
    projected to d_model, then processed by a Transformer encoder.
    Token representations are averaged and projected to embed_dim.
    """

    def __init__(self, seq_len=3750, patch_len=50, stride=25,
                 d_model=128, n_heads=8, n_layers=4,
                 dropout=0.1, embed_dim=256):
        super().__init__()
        self.patch_len = patch_len
        self.stride    = stride
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.d_model   = d_model

        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=self.n_patches + 4,
                                             dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.proj_out    = nn.Linear(d_model, embed_dim)

    def forward(self, x):   # x: (B, seq_len)
        B = x.size(0)
        # Unfold into patches
        patches = x.unfold(-1, self.patch_len, self.stride)   # (B, n_patches, patch_len)
        tokens  = self.patch_proj(patches)                     # (B, n_patches, d_model)
        tokens  = self.pos_enc(tokens)
        tokens  = self.transformer(tokens)                     # (B, n_patches, d_model)
        pooled  = tokens.mean(dim=1)                           # (B, d_model)
        return self.proj_out(pooled)                           # (B, embed_dim)


# ─────────────────────────────────────────────────────────────────────────────
# Full Multi-Channel PatchTST Classifier
# ─────────────────────────────────────────────────────────────────────────────
class MultiChannelPatchTSTClassifier(nn.Module):
    """
    Wraps one PatchTSTEncoder per channel (channel-independent encoding),
    fuses with scalar features, then outputs a single sigmoid probability.

    Args:
        n_channels  : Number of waveform channels (1, 2, or 3).
        scalar_dim  : Number of scalar features appended (default 4).
        seq_len     : Length of each waveform (default 3750).
        patch_len   : Patch length for PatchTST (default 50).
        stride      : Patch stride (default 25).
        d_model     : Transformer hidden dim (default 128).
        n_heads     : Attention heads (default 8).
        n_layers    : Transformer layers (default 4).
        embed_dim   : Per-channel output embedding dim (default 256).
        dropout     : Dropout rate (default 0.1).
    """

    def __init__(self, n_channels=3, scalar_dim=4,
                 seq_len=3750, patch_len=50, stride=25,
                 d_model=128, n_heads=8, n_layers=4,
                 embed_dim=256, dropout=0.1):
        super().__init__()
        self.n_channels = n_channels

        # Shared encoder weights across all channels (channel-independent)
        self.encoder = PatchTSTEncoder(
            seq_len=seq_len, patch_len=patch_len, stride=stride,
            d_model=d_model, n_heads=n_heads, n_layers=n_layers,
            dropout=dropout, embed_dim=embed_dim,
        )

        # Scalar MLP
        self.scalar_mlp = nn.Sequential(
            nn.Linear(scalar_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32),
        )

        # Fusion + classification head  (no MAP regression head)
        fused_dim = n_channels * embed_dim + 32
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),   # single logit → sigmoid
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, waveforms, scalars):
        """
        Args:
            waveforms : (B, n_channels, seq_len)
            scalars   : (B, scalar_dim)
        Returns:
            logit     : (B,)  raw logit (apply sigmoid for probability)
        """
        channel_embeds = []
        for i in range(self.n_channels):
            emb = self.encoder(waveforms[:, i, :])   # (B, embed_dim)
            channel_embeds.append(emb)

        fused = torch.cat(channel_embeds, dim=-1)    # (B, n_channels * embed_dim)
        sc    = self.scalar_mlp(scalars)              # (B, 32)
        fused = torch.cat([fused, sc], dim=-1)        # (B, fused_dim)

        logit = self.classifier(fused).squeeze(-1)   # (B,)
        return logit


# ─────────────────────────────────────────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

for nc in [1, 2, 3]:
    m = MultiChannelPatchTSTClassifier(n_channels=nc).to(DEVICE)
    w = torch.randn(2, nc, 3750).to(DEVICE)
    s = torch.randn(2, 4).to(DEVICE)
    out = m(w, s)
    print(f"  n_channels={nc}  →  logit shape: {out.shape}  "
          f"| params: {sum(p.numel() for p in m.parameters()):,}")

print("\nCell 2 ready.")

## CELL 3: ABLATION CONFIG — CHANNEL COMBOS × HORIZONS

In [ ]:
# ==============================================================================
# CELL 3: ABLATION CONFIGURATION
#
# Defines the full cross-product of:
#   • Channel combinations: ppg | abp | ecg | ppg+abp | abp+ecg | ppg+abp+ecg
#   • Horizons            : 5 min | 7 min | 10 min | 15 min
#
# Each (channels, horizon) pair is one ablation run.
# ==============================================================================

from itertools import product as iproduct

# ── Channel combinations to ablate (order matters: canonical is ppg, abp, ecg)
CHANNEL_COMBOS = {
    'ppg':         ['ppg'],
    'abp':         ['abp'],
    'ecg':         ['ecg'],
    'ppg+abp':     ['ppg', 'abp'],
    'abp+ecg':     ['abp', 'ecg'],
    'ppg+abp+ecg': ['ppg', 'abp', 'ecg'],
}

# ── Prediction horizons
HORIZONS = [5, 7, 10, 15]

# ── Training hyper-parameters
CONFIG = dict(
    binary_root  = '/kaggle/working/binary_data',
    batch_size   = 32,
    n_epochs     = 30,
    lr           = 1e-4,
    weight_decay = 1e-4,
    n_folds      = 5,
    pos_weight   = 3.0,   # for BCEWithLogitsLoss — adjust to dataset imbalance
    # PatchTST encoder params
    patch_len    = 50,
    stride       = 25,
    d_model      = 128,
    n_heads      = 8,
    n_layers     = 4,
    embed_dim    = 256,
    dropout      = 0.1,
)

# ── Full ablation grid
ABLATION_GRID = list(iproduct(CHANNEL_COMBOS.keys(), HORIZONS))
print(f"Total ablation runs: {len(ABLATION_GRID)}")
print(f"  = {len(CHANNEL_COMBOS)} channel combos × {len(HORIZONS)} horizons")
print()
for combo_name, horizon in ABLATION_GRID:
    print(f"  [{combo_name:>12s}  |  horizon={horizon:>2d}m]")

print("\nCell 3 ready.")

## CELL 4: TRAINING UTILITIES

In [ ]:
# ==============================================================================
# CELL 4: TRAINING UTILITIES
#
# • train_one_epoch  : single training pass with BCEWithLogitsLoss
# • evaluate         : returns loss, AUROC, AUPRC, F1, accuracy
# • run_fold         : trains one (channels, horizon, fold) triple
# • compute_metrics  : sklearn-based metric bundle
# ==============================================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, accuracy_score)
import numpy as np


def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    auroc = roc_auc_score(labels, probs)             if len(np.unique(labels)) > 1 else 0.5
    auprc = average_precision_score(labels, probs)   if len(np.unique(labels)) > 1 else 0.5
    f1    = f1_score(labels, preds, zero_division=0)
    acc   = accuracy_score(labels, preds)
    return dict(auroc=auroc, auprc=auprc, f1=f1, acc=acc)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for (waveforms, scalars), labels in loader:
        waveforms = waveforms.to(device)
        scalars   = scalars.to(device)
        labels    = labels.to(device)

        optimizer.zero_grad()
        logit = model(waveforms, scalars)
        loss  = criterion(logit, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0

    for (waveforms, scalars), labels in loader:
        waveforms = waveforms.to(device)
        scalars   = scalars.to(device)
        labels    = labels.to(device)

        logit = model(waveforms, scalars)
        loss  = criterion(logit, labels)
        total_loss += loss.item() * labels.size(0)

        probs = torch.sigmoid(logit).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = compute_metrics(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = avg_loss
    return metrics


def build_model(n_channels, cfg, device):
    model = MultiChannelPatchTSTClassifier(
        n_channels  = n_channels,
        scalar_dim  = 4,
        seq_len     = 3750,
        patch_len   = cfg['patch_len'],
        stride      = cfg['stride'],
        d_model     = cfg['d_model'],
        n_heads     = cfg['n_heads'],
        n_layers    = cfg['n_layers'],
        embed_dim   = cfg['embed_dim'],
        dropout     = cfg['dropout'],
    ).to(device)
    return model


def run_fold(fold_idx, train_idx, val_idx,
            pooled_aug, pooled_noaug,
            n_channels, cfg, device):
    """
    Train one fold. Uses augmented pool for training, non-augmented for val.
    Returns best val metrics dict.
    """
    train_subset = Subset(pooled_aug,   train_idx)
    val_subset   = Subset(pooled_noaug, val_idx)

    train_loader = DataLoader(train_subset, batch_size=cfg['batch_size'],
                              shuffle=True,  num_workers=2, pin_memory=True,
                              drop_last=True)
    val_loader   = DataLoader(val_subset,   batch_size=cfg['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    model     = build_model(n_channels, cfg, device)
    optimizer = AdamW(model.parameters(), lr=cfg['lr'],
                      weight_decay=cfg['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg['n_epochs'], eta_min=1e-6)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([cfg['pos_weight']], device=device)
    )

    best_auroc = 0.0
    best_metrics = {}

    for epoch in range(1, cfg['n_epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_m      = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if val_m['auroc'] > best_auroc:
            best_auroc   = val_m['auroc']
            best_metrics = val_m.copy()
            best_metrics['epoch'] = epoch
            # Optionally save checkpoint:
            # torch.save(model.state_dict(), f'best_fold{fold_idx}.pt')

        if epoch % 5 == 0 or epoch == 1:
            print(f"      epoch {epoch:>3d} | "
                  f"train_loss={train_loss:.4f} | "
                  f"val_auroc={val_m['auroc']:.4f} | "
                  f"val_auprc={val_m['auprc']:.4f} | "
                  f"val_f1={val_m['f1']:.4f}")

    return best_metrics, model


print("Cell 4 ready.")

## CELL 5: MULTI-HORIZON ABLATION TRAINING LOOP

In [ ]:
# ==============================================================================
# CELL 5: MULTI-HORIZON CHANNEL-ABLATION TRAINING LOOP
#
# Iterates over every (channel_combo, horizon) pair.
# For each pair: 5-fold CV → average fold metrics → test evaluation.
# All results are collected in `ablation_results` list of dicts.
# ==============================================================================

import json
import time
from collections import defaultdict

ablation_results = []   # list of result dicts, one per (combo, horizon)

for combo_name, horizon in ABLATION_GRID:
    channels   = CHANNEL_COMBOS[combo_name]
    n_channels = len(channels)

    print("=" * 70)
    print(f" ABLATION RUN: channels={combo_name}  horizon={horizon}m")
    print("=" * 70)
    t0 = time.time()

    # ── Build pooled CV datasets for this (channel, horizon) combo
    pooled_aug, pooled_noaug, pooled_labels, folds, test_ds = build_cv_pools(
        binary_root = CONFIG['binary_root'],
        horizon     = horizon,
        channels    = channels,
        n_folds     = CONFIG['n_folds'],
    )

    test_loader = DataLoader(
        test_ds,
        batch_size  = CONFIG['batch_size'],
        shuffle     = False,
        num_workers = 2,
        pin_memory  = True,
    )

    # ── 5-fold CV
    fold_metrics = defaultdict(list)
    best_fold_model = None
    best_fold_auroc = 0.0

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n  ── Fold {fold_idx + 1}/{CONFIG['n_folds']} ──")
        fold_m, fold_model = run_fold(
            fold_idx     = fold_idx,
            train_idx    = train_idx,
            val_idx      = val_idx,
            pooled_aug   = pooled_aug,
            pooled_noaug = pooled_noaug,
            n_channels   = n_channels,
            cfg          = CONFIG,
            device       = DEVICE,
        )
        for k, v in fold_m.items():
            fold_metrics[k].append(v)

        if fold_m['auroc'] > best_fold_auroc:
            best_fold_auroc = fold_m['auroc']
            best_fold_model = fold_model

    # ── Aggregate CV results
    cv_summary = {f"cv_{k}_mean": float(np.mean(v))
                  for k, v in fold_metrics.items() if k != 'epoch'}
    cv_summary.update({f"cv_{k}_std": float(np.std(v))
                       for k, v in fold_metrics.items() if k != 'epoch'})

    print(f"\n  CV Summary:")
    print(f"    AUROC : {cv_summary['cv_auroc_mean']:.4f} ± {cv_summary['cv_auroc_std']:.4f}")
    print(f"    AUPRC : {cv_summary['cv_auprc_mean']:.4f} ± {cv_summary['cv_auprc_std']:.4f}")
    print(f"    F1    : {cv_summary['cv_f1_mean']:.4f} ± {cv_summary['cv_f1_std']:.4f}")
    print(f"    ACC   : {cv_summary['cv_acc_mean']:.4f} ± {cv_summary['cv_acc_std']:.4f}")

    # ── Test evaluation using best fold model
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([CONFIG['pos_weight']], device=DEVICE)
    )
    test_m = evaluate(best_fold_model, test_loader, criterion, DEVICE)
    print(f"\n  Test Set:")
    print(f"    AUROC={test_m['auroc']:.4f}  AUPRC={test_m['auprc']:.4f}  "
          f"F1={test_m['f1']:.4f}  ACC={test_m['acc']:.4f}")

    elapsed = time.time() - t0
    print(f"  Elapsed: {elapsed/60:.1f} min")

    # ── Collect
    run_result = dict(
        combo_name = combo_name,
        channels   = channels,
        horizon    = horizon,
        n_channels = n_channels,
        elapsed_s  = elapsed,
        **cv_summary,
        **{f"test_{k}": v for k, v in test_m.items()},
    )
    ablation_results.append(run_result)

print("\n" + "=" * 70)
print(" ALL ABLATION RUNS COMPLETE")
print("=" * 70)

## CELL 6: RESULTS — SUMMARY TABLE & PLOTS

In [ ]:
# ==============================================================================
# CELL 6: ABLATION RESULTS — SUMMARY TABLE & VISUALISATIONS
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Build results dataframe
df_res = pd.DataFrame(ablation_results)
df_res = df_res.sort_values(['horizon', 'combo_name'])

# ── Print pivoted table
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(" ABLATION RESULTS — Test AUROC")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
pivot_auroc = df_res.pivot(index='combo_name', columns='horizon', values='test_auroc')
print(pivot_auroc.to_string(float_format=lambda x: f"{x:.4f}"))

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(" ABLATION RESULTS — Test AUPRC")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
pivot_auprc = df_res.pivot(index='combo_name', columns='horizon', values='test_auprc')
print(pivot_auprc.to_string(float_format=lambda x: f"{x:.4f}"))

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(" ABLATION RESULTS — Test F1")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
pivot_f1 = df_res.pivot(index='combo_name', columns='horizon', values='test_f1')
print(pivot_f1.to_string(float_format=lambda x: f"{x:.4f}"))

# ── Save results to CSV
df_res.to_csv('ablation_results.csv', index=False)
print("\n[✓] Results saved to ablation_results.csv")

# ── Figure 1: Heatmap — Test AUROC (channels × horizons)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (pivot, title) in zip(axes, [
    (pivot_auroc, 'Test AUROC'),
    (pivot_auprc, 'Test AUPRC'),
    (pivot_f1,    'Test F1'),
]):
    sns.heatmap(
        pivot, annot=True, fmt='.3f', cmap='YlOrRd',
        vmin=0.5, vmax=1.0, linewidths=0.5,
        ax=ax, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel('Horizon (min)')
    ax.set_ylabel('Channel Combination')

plt.suptitle('PatchTST Channel × Horizon Ablation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ablation_heatmaps.pdf', bbox_inches='tight')
plt.show()
print("[✓] Saved ablation_heatmaps.pdf")

# ── Figure 2: Line plots — AUROC vs horizon per channel combo
fig, ax = plt.subplots(figsize=(10, 6))
combo_order = list(CHANNEL_COMBOS.keys())
palette     = sns.color_palette('tab10', n_colors=len(combo_order))
markers     = ['o', 's', '^', 'D', 'v', 'P']

for i, combo_name in enumerate(combo_order):
    sub = df_res[df_res['combo_name'] == combo_name].sort_values('horizon')
    ax.plot(sub['horizon'], sub['test_auroc'],
            marker=markers[i], label=combo_name,
            color=palette[i], linewidth=2, markersize=8)
    # CV error band
    ax.fill_between(
        sub['horizon'],
        sub['cv_auroc_mean'] - sub['cv_auroc_std'],
        sub['cv_auroc_mean'] + sub['cv_auroc_std'],
        alpha=0.12, color=palette[i],
    )

ax.set_xlabel('Prediction Horizon (min)', fontsize=12)
ax.set_ylabel('Test AUROC', fontsize=12)
ax.set_title('AUROC vs Prediction Horizon by Channel Combination', fontsize=13, fontweight='bold')
ax.set_xticks(HORIZONS)
ax.legend(title='Channels', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('ablation_auroc_vs_horizon.pdf', bbox_inches='tight')
plt.show()
print("[✓] Saved ablation_auroc_vs_horizon.pdf")

# ── Figure 3: Bar chart — best horizon per combo
best_per_combo = df_res.loc[df_res.groupby('combo_name')['test_auroc'].idxmax()]
best_per_combo = best_per_combo.sort_values('test_auroc', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    best_per_combo['combo_name'],
    best_per_combo['test_auroc'],
    color=[palette[combo_order.index(c)] for c in best_per_combo['combo_name']],
    edgecolor='black', linewidth=0.7,
)
for bar, (_, row) in zip(bars, best_per_combo.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{row['test_auroc']:.3f}\n({row['horizon']}m)",
            ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Channel Combination', fontsize=12)
ax.set_ylabel('Best Test AUROC', fontsize=12)
ax.set_title('Best Test AUROC per Channel Combo (across horizons)', fontsize=13, fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('ablation_best_auroc_per_combo.pdf', bbox_inches='tight')
plt.show()
print("[✓] Saved ablation_best_auroc_per_combo.pdf")

print("\nCell 6 complete. All results saved.")

## CELL 7: ONE-TIME BINARY CONVERSION (RUN BEFORE TRAINING)

In [ ]:
# ==============================================================================
# CELL 7: ONE-TIME CSV → BINARY CONVERSION
#
# Run this cell ONCE before starting any training.
# After conversion, all subsequent cells read from BINARY_ROOT (much faster).
#
# Set DATA_DIR to the location of your Master_Training_Data_Expanded folder.
# Set BINARY_ROOT to a writable working directory.
# ==============================================================================

DATA_DIR    = "/kaggle/input/datasets/vmpr67/thesis-dataset-extended/Master_Training_Data_Expanded"
BINARY_ROOT = "/kaggle/working/binary_data"

conv = BinaryConverter(data_dir=DATA_DIR, binary_root=BINARY_ROOT)
conv.convert_all(
    horizons = [5, 7, 10, 15],   # Only horizons used in ablation
    splits   = ['train', 'val', 'test'],
)

print("\n[✓] Conversion done. Update CONFIG['binary_root'] and run Cell 5.")